# PyTextAD tutorial: finding anomalous words and documents

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charles-cao/pytextad/blob/main/examples/tutorial.ipynb)

This notebook detects anomalies in restaurant reviews with [PyTextAD](https://github.com/charles-cao/pytextad):
which reviews are anomalous, and which **words** make them so.

* No labelled anomalies are needed for training: the detectors learn from normal text only.
* Labels are used only to evaluate the results.
* In Colab, choose *Runtime > Change runtime type > GPU* to run faster (a CPU works too).

In [ ]:
!pip install -q pytextad pyod

## 1. Load a dataset

Every document is a list of words with a 0/1 label for each word. A document is anomalous if
any of its words is. Here, anomalous reviews have a negative sentiment.

In [ ]:
import numpy as np
from pytextad.datasets import load_dataset

ds = load_dataset("restaurant_review")
print(ds)
doc = ds[int(ds.anomalous_indices[0])]
print(doc["text"])
print([w for w, l in zip(doc["tokens"], doc["token_labels"]) if l == 1])   # the anomalous words

## 2. Split the data

Train on half of the normal reviews, test on all the others (normal and anomalous).

In [ ]:
rng = np.random.RandomState(42)
normal = ds.normal_indices
train_idx = rng.choice(normal, size=len(normal) // 2, replace=False)
train = ds.subset(train_idx)
test = ds.subset(np.setdiff1d(np.arange(len(ds)), train_idx))
print(len(train), "training reviews;", len(test), "test reviews,", int(test.labels.sum()), "anomalous")

## 3. Token-level detection

Every word is embedded with BERT (one vector per sub-word), the detector learns what normal
sub-words look like, every test sub-word gets a score, and the scores of a word's sub-words
are combined into the word's score. `evaluate` reports AUROC, average precision and the false
positive rate at 95% recall for the words, and for the reviews (word scores combined by max
and by mean), over three seeds.

In [ ]:
from pytextad import TokenCore, TokenDetector, TokenEmbedder, align_labels
from pytextad.metrics import evaluate, format_results

emb = TokenEmbedder("bert-base-uncased")
X_train = emb.transform(train.tokens)          # embed once, reuse for every detector
X_test = emb.transform(test.tokens)
y_test = align_labels(test.token_labels, X_test[1])

res = evaluate(TokenDetector(TokenCore()), X_train, X_test, token_labels=y_test)
print(format_results({"TokenCore": res}))

## 4. See which words are anomalous

The stronger the colour, the more anomalous the word. Underlined words are the ones labelled
anomalous. Hover over a word to see its score.

In [ ]:
from pytextad import highlight

det = TokenDetector(TokenCore(), embedder=emb).fit(train.tokens)
idx = np.where(test.labels == 1)[0][:5]                  # five anomalous reviews
docs = [test.tokens[i] for i in idx]
highlight(docs, det.token_scores(docs), labels=[test.token_labels[i] for i in idx])

## 5. Document-level detection

With one vector per review, any detector for vectors works directly.

In [ ]:
from pytextad import SIK, DocumentDetector, SentenceEmbedder

sent = SentenceEmbedder("bert-base-uncased", pooling="mean")
D_train, D_test = sent.transform(train.texts), sent.transform(test.texts)
res = evaluate(DocumentDetector(SIK()), D_train, D_test, doc_labels=test.labels)
print(format_results({"SIK": res}))

## 6. Compare detectors

Detectors of PyTextAD and of PyOD can be compared on the same embeddings.

In [ ]:
from pytextad import SIK
from pyod.models.ecod import ECOD
from pyod.models.iforest import IForest

detectors = {"TokenCore": TokenCore(), "SIK": SIK(), "IForest": IForest(), "ECOD": ECOD()}
results = {name: evaluate(TokenDetector(d), X_train, X_test, token_labels=y_test, seeds=(0,))
           for name, d in detectors.items()}
print(format_results(results))

## Next steps

* Other datasets: `from pytextad.datasets import list_datasets; list_datasets()`
* All detectors: [Algorithms](https://pytextad.readthedocs.io/en/latest/algorithms.html)
* Your own data: `TextADDataset.from_lists(tokens, labels)` or `load_local(path)`
* Documentation: https://pytextad.readthedocs.io